# EDA: สำรวจผล CIC-IDS2017

Notebook นี้เป็นงานสำรวจหลัง `harmonization.ipynb`; ไม่เปลี่ยนข้อมูลและไม่ทำขั้น cleansing ซ้ำ. ตารางแสดงจำนวน flow, ป้ายกำกับ, protocol/service/state และค่าสถิติ feature.


## 1. หา output และตรวจตารางที่เชื่อมด้วย record_id

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATASET_NAME = 'CIC-IDS2017'
def find_dataset_dir(start: Path, dataset_name: str) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if candidate.name == dataset_name and (candidate / "download.py").is_file():
            return candidate
        nested = candidate / "data-pipeline" / "sub-dataset" / dataset_name
        if (nested / "download.py").is_file():
            return nested
    raise FileNotFoundError(f"หาโฟลเดอร์ {dataset_name} จาก {start} ไม่พบ")

DATASET_DIR = find_dataset_dir(Path.cwd(), DATASET_NAME)
HARM_DIR = DATASET_DIR / "output" / "harmonized"
features, labels, metadata = [
    pd.read_csv(HARM_DIR / f"{name}.csv", low_memory=False)
    for name in ["features", "labels", "metadata"]
]
aligned = all(frame["record_id"].is_unique for frame in [features, labels, metadata]) and features["record_id"].tolist() == labels["record_id"].tolist() == metadata["record_id"].tolist()
print("Rows:", len(features), "| IDs aligned:", aligned)
if not aligned:
    raise ValueError("record_id ไม่ตรงกันระหว่างตาราง")


Rows: 170657 | IDs aligned: True


## 2. สำรวจป้ายกำกับและแหล่ง split

In [2]:
display(labels["attack_category"].value_counts(dropna=False).rename("rows").to_frame())
display(metadata["source_split"].value_counts(dropna=False).rename("rows").to_frame())


,rows
attack_category,
Benign / Normal,76795
Denial of Service (DoS),42664
Credential Abuse (R2L),22184
Reconnaissance & Probing,20478
"Malware, C2 & Botnet",8536


,rows
source_split,
train,150657
test,20000


## 3. สำรวจชนิด traffic และขนาดการเชื่อมต่อ

In [3]:
for column in ["protocol", "service_group", "conn_state_group"]:
    print(column)
    display(features[column].value_counts(dropna=False).rename("rows").to_frame())

numeric = [column for column in ["duration", "bytes_forward", "bytes_backward", "packets_forward", "packets_backward", "mean_packet_size"] if column in features]
display(features[numeric].describe(percentiles=[0.5, 0.9, 0.99]).T)


protocol


,rows
protocol,
tcp,147342
udp,23315


service_group


,rows
service_group,
http,85985
https_ssl,22037
other,20478
ssh,19737
dns,18189
ftp,4231


conn_state_group


,rows
conn_state_group,
established,111904
other,25556
reset,19619
no-reply,13578


,count,mean,std,min,50%,90%,99%,max
duration,170657.0,0.364369,0.509143,0.000100,0.161927,1.003224,2.371579,8.069225e+00
bytes_forward,170657.0,11332.675232,61760.959863,4.000000,895.000000,20706.000000,187279.480000,8.409117e+06
bytes_backward,170657.0,2869.466433,10060.712868,0.000000,213.000000,6977.000000,37589.320000,1.188437e+06
packets_forward,170657.0,13.382944,10.624634,1.000000,10.000000,31.000000,39.000000,5.700000e+01
packets_backward,170657.0,6.690133,6.170804,0.000000,7.000000,15.000000,21.000000,3.200000e+01
mean_packet_size,170657.0,544.281179,2099.210322,0.972973,152.333333,1070.068627,6606.747407,2.627865e+05


## 4. ตรวจรายงาน harmonization

ใช้รายงานนี้เพื่อดูขนาดตารางและ quality flags ก่อนส่ง CENTRAL builder.

In [4]:
for filename in ["harmonization_report.csv", "harmonization_quality.csv"]:
    path = HARM_DIR / filename
    if path.is_file():
        print(filename)
        display(pd.read_csv(path))


harmonization_report.csv


,table,rows,columns
0,features,170657,22
1,labels,170657,8
2,metadata,170657,35


harmonization_quality.csv


,source_dataset,split,rows,eligible_rows,quarantined_rows,invalid_required_value,bytes_packet_conflict,state_counter_conflict,invalid_duration,invalid_bytes_forward,invalid_bytes_backward,state_reclassified
0,CIC-IDS2017,test,20000,17059,2941,0,1737,2000,0,0,0,2000
1,CIC-IDS2017,train,150657,128575,22082,0,13012,15020,0,0,0,15020
